# Exercise 1: Tensor basics 
In this exercise you will learn the basics of tensor creation, manipulation, indexing, broadcasting, vectorization, einsum, and attention masking fundamentals. These basics are important for understanding any complex implementation later on so make sure you understand them well.

**To complete this exercise fill in all TODOs in the functions below.** 

Make sure to check the output of your function and whether or not it fulfills the requirements outlined in the function definition. Do NOT change the function signature or name since we will be running checks on your functions during grading.

### Shape legend used in this notebook
- `B`: batch size
- `T`: sequence length / time
- `D`: feature dimension
- `H`: number of attention heads
- `Dh`: per-head feature dimension

### Debugging tip: what to print
When you get a shape error, print:
- `x.shape`, `x.dtype`, `x.device`
- `x.is_contiguous()` (important for `view`)
For masks also print:
- `mask.shape`, `mask.dtype`, `mask.sum()` and a small slice like `mask[0, :10]`

### Reproducibility tip: seeding in PyTorch
Many operations in deep learning involve randomness (e.g., initializing model weights, shuffling data, dropout, random augmentations).
**Seeding** sets the starting state of PyTorch’s random number generator so that these random choices become **repeatable**.

- If you set the same seed and run the same code again, you should get the same *random* tensors / initial weights.
- If you don’t set a seed, results can vary between runs.

Common usage: `torch.manual_seed(seed)`

Note: even with fixed seeds, some GPU operations can still be non-deterministic due to performance optimizations. For this assignment, seeding is mainly to make debugging easier and to ensure everyone can reproduce the same intermediate results. If you are given a seed, make sure to use it when creating tensors or performing other operations.

## Tensor creation
This warmup exercise teaches you how to create tensors with different shapes and values. A few details about tensor creation that are good to know:
- `torch.tensor([...])` infers dtype from Python values (ints → integer tensor, floats → float tensor).
- `torch.arange(start, end)` is **end-exclusive**.
- `torch.linspace(start, end, steps)` is **end-inclusive**.

In [235]:
from collections.abc import Sequence
import torch

In [236]:
def make_tensor(data, dtype: torch.dtype | None = None, device: torch.device | str | None = None) -> torch.Tensor:
    """ Create a tensor from Python data (list/tuple/nested lists). """
    return torch.tensor(data=data, dtype=dtype, device=device)

# ---- build a tensor from a plain Python list ----
print("from [[1, 2], [3, 4]] with dtype=float32:")
print(make_tensor([[1, 2], [3, 4]], dtype=torch.float32).tolist())
print()
print("without dtype, the type is INFERRED from the values:")
print("  ints   [1, 2]     ->", make_tensor([1, 2]).dtype)
print("  floats [1.0, 2.0] ->", make_tensor([1.0, 2.0]).dtype)

from [[1, 2], [3, 4]] with dtype=float32:
[[1.0, 2.0], [3.0, 4.0]]

without dtype, the type is INFERRED from the values:
  ints   [1, 2]     -> torch.int64
  floats [1.0, 2.0] -> torch.float32


In [237]:
def make_zeros(shape: Sequence[int], dtype: torch.dtype | None = None, device: torch.device | str | None = None) -> torch.Tensor:
    """Create a tensor filled with zeros."""
    return torch.zeros(shape, dtype=dtype, device=device)

# ---- a tensor of zeros with the shape you ask for ----
print("make_zeros((2, 3))  ->", make_zeros((2, 3)).tolist())
print("shape", tuple(make_zeros((2, 3)).shape), " dtype", make_zeros((2, 3)).dtype, "(default)")
print()
print("dtype=float64       ->", make_zeros((2, 3), dtype=torch.float64).dtype)
print("the shape is a tuple; the values are always 0")

make_zeros((2, 3))  -> [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]]
shape (2, 3)  dtype torch.float32 (default)

dtype=float64       -> torch.float64
the shape is a tuple; the values are always 0


In [238]:
def make_ones_like(x: torch.Tensor) -> torch.Tensor:
    """Create a tensor of ones with the same shape, dtype, and device as x. """
    return torch.ones_like(x)

# ---- copy someone else's shape and dtype, but fill with ones ----
base = torch.tensor([[1.5, 2.5, 3.5],
                     [4.5, 5.5, 6.5]])
print("base        ", base.tolist(), " shape", tuple(base.shape), base.dtype)
print("ones_like ->", make_ones_like(base).tolist(), " shape", tuple(make_ones_like(base).shape), make_ones_like(base).dtype)
print("same shape, same dtype, same device -- only the values change")

base         [[1.5, 2.5, 3.5], [4.5, 5.5, 6.5]]  shape (2, 3) torch.float32
ones_like -> [[1.0, 1.0, 1.0], [1.0, 1.0, 1.0]]  shape (2, 3) torch.float32
same shape, same dtype, same device -- only the values change


In [239]:
def make_arange(start: int, end: int, step: int = 1, dtype: torch.dtype | None = None, device: torch.device | str | None = None) -> torch.Tensor:
    """Create a 1D tensor containing values [start, start+step, ..., < end]."""
    return torch.arange(start, end, step, dtype=dtype, device=device)

# ---- arange counts by a STEP, and stops BEFORE the end ----
print("make_arange(0, 5)      ->", make_arange(0, 5).tolist(), "  5 is NOT included")
print("make_arange(0, 5, 2)   ->", make_arange(0, 5, 2).tolist(), "     step of 2")
print("make_arange(2, 10, 3)  ->", make_arange(2, 10, 3).tolist(), "     starts at 2")
print()
print("you choose the STEP; how many values you get falls out of the math")

make_arange(0, 5)      -> [0, 1, 2, 3, 4]   5 is NOT included
make_arange(0, 5, 2)   -> [0, 2, 4]      step of 2
make_arange(2, 10, 3)  -> [2, 5, 8]      starts at 2

you choose the STEP; how many values you get falls out of the math


In [240]:
def make_linspace(start: float, end: float, steps: int, dtype: torch.dtype | None = None, device: torch.device | str | None = None) -> torch.Tensor:
    """Create a 1D tensor with evenly spaced values from start to end (inclusive)."""
    return torch.linspace(start, end, steps, dtype=dtype, device=device)

# ---- linspace counts a NUMBER OF POINTS, and includes both ends ----
print("make_linspace(0, 1, 5)   ->", make_linspace(0.0, 1.0, 5).tolist())
print("make_linspace(0, 10, 3)  ->", make_linspace(0.0, 10.0, 3).tolist())
print()
print("compare the two on the same range 0 -> 1:")
print("  arange(0, 1, 0.25)  ->", make_arange(0, 1, 0.25).tolist(), "  4 values, 1.0 missing")
print("  linspace(0, 1, 5)   ->", make_linspace(0.0, 1.0, 5).tolist(), " 5 values, 1.0 included")

make_linspace(0, 1, 5)   -> [0.0, 0.25, 0.5, 0.75, 1.0]
make_linspace(0, 10, 3)  -> [0.0, 5.0, 10.0]

compare the two on the same range 0 -> 1:
  arange(0, 1, 0.25)  -> [0.0, 0.25, 0.5, 0.75]   4 values, 1.0 missing
  linspace(0, 1, 5)   -> [0.0, 0.25, 0.5, 0.75, 1.0]  5 values, 1.0 included


In [241]:
def make_randn(shape: Sequence[int], seed: int | None = None, dtype: torch.dtype | None = None, device: torch.device | str | None = None) -> torch.Tensor:
    """Create a tensor filled with values from a standard normal distribution."""
    generator = None
    if seed is not None:
        generator = torch.Generator(device=device or "cpu")
        generator.manual_seed(seed)
    return torch.randn(shape, generator=generator, dtype=dtype, device=device)

# ---- random values from a bell curve centred on 0 ----
def show(t):
    return [[round(v, 3) for v in row] for row in t.tolist()]

print("seed=123 ->", show(make_randn((2, 3), seed=123)))
print("seed=123 ->", show(make_randn((2, 3), seed=123)), " <- same seed, same numbers")
print("seed=999 ->", show(make_randn((2, 3), seed=999)), " <- different seed")
print()
big = make_randn((100000,), seed=0)
print("over 100k samples:  mean %.3f   std %.3f   (a standard normal: mean 0, std 1)"
      % (big.mean(), big.std()))

seed=123 -> [[-0.111, 0.12, -0.37], [-0.24, -1.197, 0.209]]
seed=123 -> [[-0.111, 0.12, -0.37], [-0.24, -1.197, 0.209]]  <- same seed, same numbers
seed=999 -> [[-0.838, 0.456, 0.348], [-0.137, 0.22, 0.983]]  <- different seed

over 100k samples:  mean -0.002   std 1.001   (a standard normal: mean 0, std 1)


In [242]:
def cast_dtype_and_move(x: torch.Tensor, device: torch.device, dtype: torch.dtype) -> torch.Tensor:
    """Convert tensor dtype and move to device."""
    return x.to(device=device, dtype=dtype)

# ---- change dtype and device in one step ----
src = torch.tensor([1, 2, 3])
out = cast_dtype_and_move(src, torch.device("cpu"), torch.float32)

print("in    ", src.tolist(), " dtype", src.dtype, " device", src.device)
print("out ->", out.tolist(), " dtype", out.dtype, " device", out.device)
print("the values are unchanged; only how they are stored changed")

in     [1, 2, 3]  dtype torch.int64  device cpu
out -> [1.0, 2.0, 3.0]  dtype torch.float32  device cpu
the values are unchanged; only how they are stored changed


## Shape manipulation
Now that we covered the basic tensor creation schemes, we want to focus on shape manipulation. Understanding the difference between these mechanisms is key for building larger systems and many people still get it wrong. 
The core ideas to understand are:
- **Contiguous tensors** store data in a single, row-major memory layout.
- Many ops (especially slicing like `x[:, ::2]`, `transpose`, `permute`) often create **non-contiguous** tensors (no copy but different strides).
- `view(...)` is **zero-copy** but typically requires **contiguous** memory → may throw an error.
- `reshape(...)` tries to return a view, but if the tensor is non-contiguous it will **allocate/copy**.
- `contiguous()` forces a contiguous copy when the tensor isn’t contiguous.

If you *need* a view after reordering dims: call `x = x.contiguous()` first (this makes a contiguous copy).

In [243]:
def reshape_tensor(x: torch.Tensor, new_shape: Sequence[int]) -> torch.Tensor:
    """Reshape tensor to new_shape (may return a view or a copy)."""
    return x.reshape(new_shape)

# ---- reshape: same numbers, new shape ----
x = torch.arange(6)
print("x               ", x.tolist(), "        shape", tuple(x.shape))
print("reshape (2,3) ->", reshape_tensor(x, (2, 3)).tolist(), "  shape (2, 3)")
print("reshape (3,2) ->", reshape_tensor(x, (3, 2)).tolist(), "  shape (3, 2)")
print()
print("the total number of elements never changes: 6 = 2*3 = 3*2")

x                [0, 1, 2, 3, 4, 5]         shape (6,)
reshape (2,3) -> [[0, 1, 2], [3, 4, 5]]   shape (2, 3)
reshape (3,2) -> [[0, 1], [2, 3], [4, 5]]   shape (3, 2)

the total number of elements never changes: 6 = 2*3 = 3*2


In [244]:
def view_tensor(x: torch.Tensor, new_shape: Sequence[int]) -> torch.Tensor:
    """View tensor as new_shape (requires contiguous memory and doesn't allocate new memory for the tensor data)."""
    return x.view(new_shape)

# ---- view: like reshape, but it REFUSES when memory does not allow it ----
print("view (2,3) ->", view_tensor(x, (2, 3)).tolist())
print("shares memory with x:", view_tensor(x, (2, 3)).data_ptr() == x.data_ptr())
print()
t = torch.arange(6).reshape(2, 3).t()      # transposed -> non-contiguous
print("after a transpose, view cannot flatten it:")
try:
    view_tensor(t, (6,))
    print("  ...succeeded")
except RuntimeError:
    print("  RuntimeError   <- this is when you need .contiguous() or reshape")
print("  reshape works ->", reshape_tensor(t, (6,)).tolist())

view (2,3) -> [[0, 1, 2], [3, 4, 5]]
shares memory with x: True

after a transpose, view cannot flatten it:
  RuntimeError   <- this is when you need .contiguous() or reshape
  reshape works -> [0, 3, 1, 4, 2, 5]


In [245]:
def flatten_from_dim(x: torch.Tensor, start_dim: int = 0) -> torch.Tensor:
    """Flatten a tensor starting from start_dim into a single dimension."""
    return x.flatten(start_dim=start_dim)

# ---- flatten: squash everything from start_dim onward into one axis ----
x2 = torch.arange(8).reshape(2, 2, 2)
print("x2                 ", x2.tolist(), " shape", tuple(x2.shape))
print("start_dim=1     -> ", flatten_from_dim(x2, 1).tolist(), "       shape (2, 4)")
print("start_dim=0     -> ", flatten_from_dim(x2, 0).tolist(), "  shape (8,)")
print()
print("start_dim=1 keeps the batch axis and flattens the rest -- the usual case")

x2                  [[[0, 1], [2, 3]], [[4, 5], [6, 7]]]  shape (2, 2, 2)
start_dim=1     ->  [[0, 1, 2, 3], [4, 5, 6, 7]]        shape (2, 4)
start_dim=0     ->  [0, 1, 2, 3, 4, 5, 6, 7]   shape (8,)

start_dim=1 keeps the batch axis and flattens the rest -- the usual case


In [246]:
def add_singleton_dim(x: torch.Tensor, dim: int) -> torch.Tensor:
    """Insert a size-1 dimension at position dim."""
    return x.unsqueeze(dim)

# ---- unsqueeze: insert an axis of length 1 (adds a layer of brackets) ----
x3 = torch.tensor([1, 2, 3])
print("x3            ", x3.tolist(), "     shape", tuple(x3.shape))
print("dim=0      -> ", add_singleton_dim(x3, 0).tolist(), "   shape (1, 3)  one row")
print("dim=1      -> ", add_singleton_dim(x3, 1).tolist(), "  shape (3, 1)  one column")
print()
print("same 3 numbers either way -- only the nesting changed")

x3             [1, 2, 3]      shape (3,)
dim=0      ->  [[1, 2, 3]]    shape (1, 3)  one row
dim=1      ->  [[1], [2], [3]]   shape (3, 1)  one column

same 3 numbers either way -- only the nesting changed


In [247]:
def remove_singleton_dims(x: torch.Tensor, dim: int | None = None) -> torch.Tensor:
    """Remove size-1 dimensions."""
    if dim is None:
        return x.squeeze()
    return x.squeeze(dim)

# ---- squeeze: drop axes of length 1 (removes brackets) ----
y = torch.tensor([[[1, 2, 3]]])
print("y                 ", y.tolist(), " shape", tuple(y.shape))
print("squeeze all    -> ", remove_singleton_dims(y).tolist(), "       shape", tuple(remove_singleton_dims(y).shape))
print("squeeze dim=0  -> ", remove_singleton_dims(y, 0).tolist(), "     shape", tuple(remove_singleton_dims(y, 0).shape))
print()
z = torch.tensor([[1, 2, 3]])
print("an axis that is NOT 1 is left alone:")
print("  z", z.tolist(), "squeeze dim=1 ->", remove_singleton_dims(z, 1).tolist(), "(unchanged)")

y                  [[[1, 2, 3]]]  shape (1, 1, 3)
squeeze all    ->  [1, 2, 3]        shape (3,)
squeeze dim=0  ->  [[1, 2, 3]]      shape (1, 3)

an axis that is NOT 1 is left alone:
  z [[1, 2, 3]] squeeze dim=1 -> [[1, 2, 3]] (unchanged)


In [248]:
def transpose_last_two(x: torch.Tensor) -> torch.Tensor:
    """Swap the last two dimensions of x."""
    return x.transpose(-2, -1)

# ---- swap the last two axes (works whatever the leading axes are) ----
x6 = torch.tensor([[[1, 2, 3],
                    [4, 5, 6]]])
print("x6            ", x6.tolist(), " shape", tuple(x6.shape))
print("transposed -> ", transpose_last_two(x6).tolist(), " shape", tuple(transpose_last_two(x6).shape))
print("               rows became columns")
print()
print("still contiguous?", transpose_last_two(x6).is_contiguous(), " <- transposing breaks contiguity")

x6             [[[1, 2, 3], [4, 5, 6]]]  shape (1, 2, 3)
transposed ->  [[[1, 4], [2, 5], [3, 6]]]  shape (1, 3, 2)
               rows became columns

still contiguous? False  <- transposing breaks contiguity


In [249]:
def permute_bhwc_to_bchw(x: torch.Tensor) -> torch.Tensor:
    """Convert (B, H, W, C) tensor into (B, C, H, W)."""
    return x.permute(0, 3, 1, 2)

# ---- image layout: channels-last -> channels-first ----
# one 2x2 image, 3 colour channels. each innermost triple is ONE pixel's (R,G,B)
img = torch.tensor([[[[1, 2, 3], [4, 5, 6]],
                     [[7, 8, 9], [10, 11, 12]]]])
print("BHWC", tuple(img.shape), " grouped by PIXEL:")
print("   row 0:", img[0, 0].tolist(), "  <- pixel(0,0)=1,2,3   pixel(0,1)=4,5,6")
print("   row 1:", img[0, 1].tolist())
print()
out = permute_bhwc_to_bchw(img)
print("BCHW", tuple(out.shape), " grouped by CHANNEL:")
for c in range(3):
    print("   channel", c, "=", out[0, c].tolist())
print()
print("same 12 numbers, regrouped. conv layers expect the second form.")

BHWC (1, 2, 2, 3)  grouped by PIXEL:
   row 0: [[1, 2, 3], [4, 5, 6]]   <- pixel(0,0)=1,2,3   pixel(0,1)=4,5,6
   row 1: [[7, 8, 9], [10, 11, 12]]

BCHW (1, 3, 2, 2)  grouped by CHANNEL:
   channel 0 = [[1, 4], [7, 10]]
   channel 1 = [[2, 5], [8, 11]]
   channel 2 = [[3, 6], [9, 12]]

same 12 numbers, regrouped. conv layers expect the second form.


In [250]:
def make_contiguous(x: torch.Tensor) -> torch.Tensor:
    """Check if tensor is contiguous and if not make contiguous."""
    return x.contiguous()

# ---- contiguous: force a tidy memory copy when the layout got scrambled ----
x8 = torch.arange(24).reshape(4, 6)[:, ::2]     # take every 2nd column
print("x8            ", x8.tolist())
print("contiguous?   ", x8.is_contiguous(), "   strides", x8.stride())
print()
x8c = make_contiguous(x8)
print("after ->      ", x8c.tolist(), "  same values")
print("contiguous?   ", x8c.is_contiguous(), "    strides", x8c.stride())
print("nothing visible changed -- only the memory layout did")

x8             [[0, 2, 4], [6, 8, 10], [12, 14, 16], [18, 20, 22]]
contiguous?    False    strides (6, 2)

after ->       [[0, 2, 4], [6, 8, 10], [12, 14, 16], [18, 20, 22]]   same values
contiguous?    True     strides (3, 1)
nothing visible changed -- only the memory layout did


## Indexing
Now that we know how to create tensors and manipulate them we need to understand how we can extract certain components from them using indexing. 
- Basic slicing (`x[a:b]`) returns a view when possible.
- “Fancy” indexing (lists/tensors of indices) usually allocates a new tensor.
- In-place vs out-of-place matters: if a function says “return a copy, leave the input unchanged”, you need `clone()`.

In [251]:
def slice_rows(x: torch.Tensor, start: int, end: int) -> torch.Tensor:
    """Slice rows in a 2D tensor: x[start:end, :]."""
    return x[start:end, :]

# ---- slice rows: x[start:end, :] ----
x = torch.arange(12).reshape(4, 3)
print("x            ", x.tolist())
print("rows 1 to 3 ->", slice_rows(x, 1, 3).tolist(), "   end is EXCLUSIVE: rows 1 and 2")
print("rows 0 to 1 ->", slice_rows(x, 0, 1).tolist(), "                just row 0")

x             [[0, 1, 2], [3, 4, 5], [6, 7, 8], [9, 10, 11]]
rows 1 to 3 -> [[3, 4, 5], [6, 7, 8]]    end is EXCLUSIVE: rows 1 and 2
rows 0 to 1 -> [[0, 1, 2]]                 just row 0


In [252]:
def select_columns(x: torch.Tensor, cols: Sequence[int]) -> torch.Tensor:
    """Select specific columns from a 2D tensor."""
    return x[:, cols]

# ---- select columns: the ':' keeps every row ----
print("x                  ", x.tolist())
print("columns [0, 2]  -> ", select_columns(x, [0, 2]).tolist())
print("                    every row kept, only columns 0 and 2 survive")
print()
print("careful -- x[[0, 2]] without the ':' would select ROWS instead:")
print("   x[[0, 2]]     ->", x[[0, 2]].tolist())

x                   [[0, 1, 2], [3, 4, 5], [6, 7, 8], [9, 10, 11]]
columns [0, 2]  ->  [[0, 2], [3, 5], [6, 8], [9, 11]]
                    every row kept, only columns 0 and 2 survive

careful -- x[[0, 2]] without the ':' would select ROWS instead:
   x[[0, 2]]     -> [[0, 1, 2], [6, 7, 8]]


In [253]:
def get_diagonal(x: torch.Tensor) -> torch.Tensor:
    """Get the diagonal of a 2D tensor."""
    return x.diag()

# ---- the diagonal: elements where row index == column index ----
m = torch.tensor([[1, 2],
                  [3, 4]])
print("m           ", m.tolist())
print("diagonal -> ", get_diagonal(m).tolist(), "   m[0][0]=1 and m[1][1]=4")
print()
big = torch.tensor([[1, 2, 3],
                    [4, 5, 6],
                    [7, 8, 9]])
print("m3          ", big.tolist())
print("diagonal -> ", get_diagonal(big).tolist())

m            [[1, 2], [3, 4]]
diagonal ->  [1, 4]    m[0][0]=1 and m[1][1]=4

m3           [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
diagonal ->  [1, 5, 9]


In [254]:
def set_subtensor(x: torch.Tensor, row_idx: int, col_idx: int, value: float) -> torch.Tensor:
    """Return a copy of x where x[row_idx, col_idx] is set to value."""
    out = x.clone()
    out[row_idx, col_idx] = value
    return out

# ---- write one element, but into a COPY ----
base = torch.zeros(2, 2)
out = set_subtensor(base, 0, 1, 5.0)

print("base before  ", base.tolist())
print("out       -> ", out.tolist(), "   position [0][1] set to 5")
print("base after   ", base.tolist(), "   unchanged -- that is what clone() buys")

base before   [[0.0, 0.0], [0.0, 0.0]]
out       ->  [[0.0, 5.0], [0.0, 0.0]]    position [0][1] set to 5
base after    [[0.0, 0.0], [0.0, 0.0]]    unchanged -- that is what clone() buys


In [255]:
def gather_rows(x: torch.Tensor, row_indices: torch.Tensor) -> torch.Tensor:
    """Gather (concat) rows from x using row_indices."""
    return x[row_indices]

# ---- pick rows by index, in any order, as many times as you like ----
x2 = torch.tensor([[10, 11],
                   [20, 21],
                   [30, 31]])
print("x2                 ", x2.tolist(), "  rows 0, 1, 2")
print("rows [2, 0]     -> ", gather_rows(x2, torch.tensor([2, 0])).tolist(), "        your order, not sorted")
print("rows [1, 1, 0]  -> ", gather_rows(x2, torch.tensor([1, 1, 0])).tolist(), " repeats are fine")
print()
print("the output length follows the INDEX list, not the input")

x2                  [[10, 11], [20, 21], [30, 31]]   rows 0, 1, 2
rows [2, 0]     ->  [[30, 31], [10, 11]]         your order, not sorted
rows [1, 1, 0]  ->  [[20, 21], [20, 21], [10, 11]]  repeats are fine

the output length follows the INDEX list, not the input


## Broadcasting and reducing
Now we're covering a pytorch mechanism that lets you apply elementwise ops without using python loops. It's important to understand how it works to trace your shapes in complicated systems. The broadcasting rules to know are:
- Dimensions align from the **right**.
- A dimension can broadcast if it’s equal or one of them is **1**.

### Reduction ops and `keepdim`

When you reduce over a dimension (e.g. `sum`, `mean`, `max`), PyTorch can either:

- **remove** the reduced dimension (`keepdim=False`, default), or
- **keep** it as size 1 (`keepdim=True`)

Keeping the dimension is often helpful because it makes broadcasting back “just work”.

#### Shape diagram examples

Assume `x` has shape `(B, T, D)`:

**Sum over time**
- `x.sum(dim=1)` → shape `(B, D)`
- `x.sum(dim=1, keepdim=True)` → shape `(B, 1, D)`

**Mean over features**
- `x.mean(dim=2)` → shape `(B, T)`
- `x.mean(dim=2, keepdim=True)` → shape `(B, T, 1)`

#### Why `keepdim=True` helps with broadcasting

Example: center `x` by subtracting the mean over `T`

- If `m = x.mean(dim=1)` has shape `(B, D)`, then `x - m` **fails** (shapes `(B,T,D)` and `(B,D)` don't align).
- If `m = x.mean(dim=1, keepdim=True)` has shape `(B,1,D)`, then `x - m` **works** via broadcasting.

In [256]:
def sum_over_dim(x: torch.Tensor, dim: int, keepdim: bool = False) -> torch.Tensor:
    """Sum tensor values along dimension dim."""
    return x.sum(dim=dim, keepdim=keepdim)

# ---- sum along one axis; keepdim decides whether the axis survives ----
x = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])
print("x                       ", x.tolist())
print("sum dim=1            -> ", sum_over_dim(x, 1).tolist(), "      shape", tuple(sum_over_dim(x, 1).shape), " axis gone")
print("sum dim=1 keepdim    -> ", sum_over_dim(x, 1, True).tolist(), "  shape", tuple(sum_over_dim(x, 1, True).shape), " axis kept as 1")
print("sum dim=0            -> ", sum_over_dim(x, 0).tolist(), "   down the columns")

x                        [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]]
sum dim=1            ->  [6.0, 15.0]       shape (2,)  axis gone
sum dim=1 keepdim    ->  [[6.0], [15.0]]   shape (2, 1)  axis kept as 1
sum dim=0            ->  [5.0, 7.0, 9.0]    down the columns


In [257]:
def mean_over_dim(x: torch.Tensor, dim: int, keepdim: bool = False) -> torch.Tensor:
    """Mean along dimension dim."""
    return x.mean(dim=dim, keepdim=keepdim)

# ---- mean works the same way; keepdim is what makes broadcasting work ----
x2 = torch.tensor([[1., 2.],
                   [3., 4.]])
print("x2                    ", x2.tolist())
print("mean dim=0         -> ", mean_over_dim(x2, 0).tolist(), "   column averages")
print("mean dim=1         -> ", mean_over_dim(x2, 1).tolist(), "   row averages")
print("mean dim=1 keepdim -> ", mean_over_dim(x2, 1, True).tolist(), " shape (2, 1)")
print()
print("why keepdim matters -- subtracting the row mean from every row:")
print("   x2 - mean(keepdim) ->", (x2 - mean_over_dim(x2, 1, True)).tolist(), " works")
print("   with keepdim=False the shapes would not line up")

x2                     [[1.0, 2.0], [3.0, 4.0]]
mean dim=0         ->  [2.0, 3.0]    column averages
mean dim=1         ->  [1.5, 3.5]    row averages
mean dim=1 keepdim ->  [[1.5], [3.5]]  shape (2, 1)

why keepdim matters -- subtracting the row mean from every row:
   x2 - mean(keepdim) -> [[-0.5, 0.5], [-0.5, 0.5]]  works
   with keepdim=False the shapes would not line up


In [258]:
def max_over_dim(x: torch.Tensor, dim: int) -> tuple[torch.Tensor, torch.Tensor]:
    """Max values and argmax indices along dimension dim."""
    values, indices = x.max(dim=dim)
    return values, indices

# ---- max returns BOTH the largest value and where it was ----
x3 = torch.tensor([[1., 5., 2.],
                   [9., 3., 4.]])
values, indices = max_over_dim(x3, dim=1)
print("x3           ", x3.tolist())
print("values    -> ", values.tolist(), "   the biggest number in each row")
print("indices   -> ", indices.tolist(), "     where it sits in that row")
print("             5 is at position 1 of row 0;  9 is at position 0 of row 1")

x3            [[1.0, 5.0, 2.0], [9.0, 3.0, 4.0]]
values    ->  [5.0, 9.0]    the biggest number in each row
indices   ->  [1, 0]      where it sits in that row
             5 is at position 1 of row 0;  9 is at position 0 of row 1


In [259]:
def argmax_over_dim(x: torch.Tensor, dim: int) -> torch.Tensor:
    """Argmax indices along dimension dim."""
    return x.argmax(dim=dim)

# ---- argmax returns only the position ----
print("x3           ", x3.tolist())
print("argmax    -> ", argmax_over_dim(x3, dim=1).tolist(), "  same indices as above, values dropped")
print()
scores = torch.tensor([[0.1, 3.2, 0.5, 0.2]])
names = ["cat", "dog", "bird", "fish"]
pick = argmax_over_dim(scores, dim=1)
print("scores       ", [round(v, 2) for v in scores[0].tolist()])
print("argmax    -> ", pick.tolist(), "-> predicted class:", names[pick.item()])
print("             this is how a classifier turns scores into an answer")

x3            [[1.0, 5.0, 2.0], [9.0, 3.0, 4.0]]
argmax    ->  [1, 0]   same indices as above, values dropped

scores        [0.1, 3.2, 0.5, 0.2]
argmax    ->  [1] -> predicted class: dog
             this is how a classifier turns scores into an answer


In [260]:
def broadcast_add_vector(x: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    """Add a vector v to each row of a 2D tensor x using broadcasting."""
    return x + v

# ---- add one vector to every row, with no loop ----
x4 = torch.tensor([[0., 0.],
                   [1., 1.],
                   [2., 2.]])
v = torch.tensor([10., 20.])
print("x4  (3,2)   ", x4.tolist())
print("v   (2,)    ", v.tolist())
print("x4 + v   -> ", broadcast_add_vector(x4, v).tolist())
print("             v was reused for all 3 rows -- 10 to column 0, 20 to column 1")
print()
print("shapes align from the RIGHT: (3,2) vs (2,) -> the 2s match, so v stretches over the 3")

x4  (3,2)    [[0.0, 0.0], [1.0, 1.0], [2.0, 2.0]]
v   (2,)     [10.0, 20.0]
x4 + v   ->  [[10.0, 20.0], [11.0, 21.0], [12.0, 22.0]]
             v was reused for all 3 rows -- 10 to column 0, 20 to column 1

shapes align from the RIGHT: (3,2) vs (2,) -> the 2s match, so v stretches over the 3


## Vectorization
We want to avoid slow (due to per-iteration overhead) python loops as much as possible and pytorch gives us many tools to avoid it. We cover these basics:
- `cat` vs `stack` (concatenate existing dims vs create a new dim)
- `repeat` vs `expand`
- `scatter_add` / `index_add` for accumulation
- `where` for conditional selection

### `expand` vs `repeat`

- `repeat(...)` **copies** data → larger tensor with independent storage.
- `expand(...)` **does not copy** data → it creates a *view* with clever strides.

This has two important implications:

1) `expand` only works when expanding a **size-1 dimension** (broadcasting a singleton).
2) The expanded tensor may have **many positions pointing to the same memory**.  
   Modifying the expanded tensor can therefore produce surprising results (multiple rows change).

Rule of thumb:
- Use `expand` for read-only broadcasting.
- Use `repeat` if you truly need independent copies.


NOTE: We implore you to write your own quick checks from now on for calling the functions and checking their output. As before you are still required to fill in the TODOs in each function.

In [261]:
def concat_tensors(tensors: Sequence[torch.Tensor], dim: int = 0) -> torch.Tensor:
    """Concatenate tensors along dim. NOTE: This will always allocate new memory"""
    return torch.cat(list(tensors), dim=dim)

# ---- glue tensors together along an axis that ALREADY exists ----
a = torch.tensor([[1, 2],
                  [3, 4]])
b = torch.tensor([[5, 6]])
c = torch.tensor([[7],
                  [8]])

print("a            ", a.tolist())
print("b            ", b.tolist())
print("cat dim=0 -> ", concat_tensors([a, b], dim=0).tolist(), "  b added as a new ROW")
print()
print("c            ", c.tolist())
print("cat dim=1 -> ", concat_tensors([a, c], dim=1).tolist(), "  c added as a new COLUMN")

a             [[1, 2], [3, 4]]
b             [[5, 6]]
cat dim=0 ->  [[1, 2], [3, 4], [5, 6]]   b added as a new ROW

c             [[7], [8]]
cat dim=1 ->  [[1, 2, 7], [3, 4, 8]]   c added as a new COLUMN


In [262]:
def stack_tensors(tensors: Sequence[torch.Tensor], dim: int = 0) -> torch.Tensor:
    """Stack tensors along a new dimension dim."""
    return torch.stack(list(tensors), dim=dim)

# ---- stack builds a NEW axis; cat does not ----
a = torch.tensor([1, 2])
b = torch.tensor([3, 4])

print("a              ", a.tolist())
print("b              ", b.tolist())
print()
print("cat   dim=0 -> ", concat_tensors([a, b], dim=0).tolist(), " shape (4,)   one long row")
print("stack dim=0 -> ", stack_tensors([a, b], dim=0).tolist(), " shape (2,2)  a and b are now rows")
print("stack dim=1 -> ", stack_tensors([a, b], dim=1).tolist(), " shape (2,2)  a and b are now columns")

a               [1, 2]
b               [3, 4]

cat   dim=0 ->  [1, 2, 3, 4]  shape (4,)   one long row
stack dim=0 ->  [[1, 2], [3, 4]]  shape (2,2)  a and b are now rows
stack dim=1 ->  [[1, 3], [2, 4]]  shape (2,2)  a and b are now columns


In [263]:
def repeat_tensor(x: torch.Tensor, repeats: Sequence[int]) -> torch.Tensor:
    """Repeat tensor along each dimension."""
    return x.repeat(*repeats)

# ---- repeat: make real copies, tiling the data ----
x = torch.tensor([[1, 2]])

print("x                ", x.tolist(), "   shape", tuple(x.shape))
print("repeat (2, 3) -> ", repeat_tensor(x, (2, 3)).tolist())
print("                  ^ 2 copies downward, 3 copies across  -> shape (2, 6)")

x                 [[1, 2]]    shape (1, 2)
repeat (2, 3) ->  [[1, 2, 1, 2, 1, 2], [1, 2, 1, 2, 1, 2]]
                  ^ 2 copies downward, 3 copies across  -> shape (2, 6)


In [264]:
def expand_tensor(x: torch.Tensor, *sizes: int) -> torch.Tensor:
    """Expand tensor to a larger size without copying data.(Sizes can be -1 to keep original dimension.)"""
    return x.expand(*sizes)

# ---- expand: stretch a size-1 axis WITHOUT copying ----
x = torch.tensor([[1],
                  [2]])

print("x               ", x.tolist(), "  shape", tuple(x.shape))
print("expand(2, 3) -> ", expand_tensor(x, 2, 3).tolist(), "  the single column shown 3 times")
print()
print("no new memory:", expand_tensor(x, 2, 3).data_ptr() == x.data_ptr())
print("only a size-1 axis can stretch. 2 -> 3 is not allowed:")
try:
    expand_tensor(torch.tensor([[1, 2]]), 1, 3)
    print("  ...succeeded")
except RuntimeError:
    print("  RuntimeError")

x                [[1], [2]]   shape (2, 1)
expand(2, 3) ->  [[1, 1, 1], [2, 2, 2]]   the single column shown 3 times

no new memory: True
only a size-1 axis can stretch. 2 -> 3 is not allowed:
  RuntimeError


In [265]:
def cumsum_over_dim(x: torch.Tensor, dim: int = 0) -> torch.Tensor:
    """Cumulative sum along dim."""
    return x.cumsum(dim=dim)

# ---- cumsum: running total, nothing collapses ----
x = torch.tensor([1, 2, 3, 4])

print("x         ", x.tolist())
print("cumsum -> ", cumsum_over_dim(x, 0).tolist())
print("            1, 1+2, 1+2+3, 1+2+3+4")
print()
m = torch.tensor([[1, 2],
                  [3, 4]])
print("m          ", m.tolist())
print("dim=0   -> ", cumsum_over_dim(m, 0).tolist(), "  running down the columns")
print("dim=1   -> ", cumsum_over_dim(m, 1).tolist(), "  running across the rows")

x          [1, 2, 3, 4]
cumsum ->  [1, 3, 6, 10]
            1, 1+2, 1+2+3, 1+2+3+4

m           [[1, 2], [3, 4]]
dim=0   ->  [[1, 2], [4, 6]]   running down the columns
dim=1   ->  [[1, 3], [3, 7]]   running across the rows


In [266]:
def where_select(mask: torch.Tensor, a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """Elementwise select: return a where mask is True else b. mask must be broadcastable to a and b."""
    return torch.where(mask, a, b)

# ---- where: pick from a or b, element by element ----
mask = torch.tensor([True, False, True])
a    = torch.tensor([1, 2, 3])
b    = torch.tensor([10, 20, 30])

print("mask   ", mask.tolist())
print("a      ", a.tolist(), "   taken where mask is True")
print("b      ", b.tolist(), " taken where mask is False")
print("out -> ", where_select(mask, a, b).tolist())
print("         ^a   ^b  ^a")

mask    [True, False, True]
a       [1, 2, 3]    taken where mask is True
b       [10, 20, 30]  taken where mask is False
out ->  [1, 20, 3]
         ^a   ^b  ^a


In [267]:
def one_hot(indices: torch.Tensor, num_classes: int, dtype: torch.dtype | None = None) -> torch.Tensor:
    """
    Create one-hot encodings.
    Output is a tensor of the same shape as indices with an added dimension of size num_classes at the end, 
    where the value along that dimension is 1 if it matches the index and 0 otherwise.

    Shapes:
    - indices: (...,) integer tensor
    Return:
    - out: (..., num_classes)

    Requirements:
    - Must work for arbitrary leading shape.
    - No Python loops.
    """
    classes = torch.arange(num_classes, device=indices.device)
    out = indices.unsqueeze(-1) == classes
    return out.to(torch.long if dtype is None else dtype)

# ---- one_hot: turn a class id into a position ----
ids = torch.tensor([0, 2, 1])

print("ids        ", ids.tolist())
print("one_hot -> ")
for i, row in zip(ids.tolist(), one_hot(ids, num_classes=3).tolist()):
    print("   id", i, "->", row)
print("the 1 sits at the position named by the id; everything else is 0")

ids         [0, 2, 1]
one_hot -> 
   id 0 -> [1, 0, 0]
   id 2 -> [0, 0, 1]
   id 1 -> [0, 1, 0]
the 1 sits at the position named by the id; everything else is 0


In [268]:
def scatter_add_1d(
    values: torch.Tensor, indices: torch.Tensor, size: int
) -> torch.Tensor:
    """
    Sum `values` into an output vector at positions `indices`.

    Shapes:
    - values: (N,)
    - indices: (N,) integer indices in [0, size)
    Return:
    - out: (size,) with same dtype and device as values

    Requirement:
    - no Python loops
    """
    out = torch.zeros(size, dtype=values.dtype, device=values.device)
    return out.index_add_(0, indices, values)

# ---- scatter_add: drop values into slots, adding on collision ----
values  = torch.tensor([1., 2., 3., 4.])
indices = torch.tensor([0,  1,  0,  2])

print("values   ", values.tolist())
print("indices  ", indices.tolist(), "  <- which slot each value goes to")
print("out ->   ", scatter_add_1d(values, indices, size=3).tolist())
print()
print("   slot 0 <- 1 + 3 = 4    (index 0 appears twice, so they ADD)")
print("   slot 1 <- 2")
print("   slot 2 <- 4")

values    [1.0, 2.0, 3.0, 4.0]
indices   [0, 1, 0, 2]   <- which slot each value goes to
out ->    [4.0, 2.0, 4.0]

   slot 0 <- 1 + 3 = 4    (index 0 appears twice, so they ADD)
   slot 1 <- 2
   slot 2 <- 4


In [269]:
def batched_token_histogram(tokens: torch.Tensor, vocab_size: int) -> torch.Tensor:
    """
    Count token occurrences per batch item.

    Shapes:
    - tokens: (B, T) int64
    Return:
    - counts: (B, vocab_size) where counts[b, v] = number of times token v appears in tokens[b] 

    Requirements:
    - No Python loops over B or T.
    """
    counts = torch.zeros(
        tokens.shape[0], vocab_size, dtype=torch.long, device=tokens.device
    )
    return counts.scatter_add_(1, tokens, torch.ones_like(tokens))

# ---- histogram: how many times does each token appear, per row ----
tokens = torch.tensor([[0, 1, 1],
                       [2, 2, 2]])

print("tokens     ", tokens.tolist())
print("counts ->  ", batched_token_histogram(tokens, vocab_size=3).tolist())
print()
print("   row 0 holds one 0, two 1s, zero 2s  ->  [1, 2, 0]")
print("   row 1 holds zero 0s, zero 1s, three 2s -> [0, 0, 3]")

tokens      [[0, 1, 1], [2, 2, 2]]
counts ->   [[1, 2, 0], [0, 0, 3]]

   row 0 holds one 0, two 1s, zero 2s  ->  [1, 2, 0]
   row 1 holds zero 0s, zero 1s, three 2s -> [0, 0, 3]


In [270]:
def masked_mean(x: torch.Tensor, mask: torch.Tensor, dim: int) -> torch.Tensor:
    """
    Mean over `dim` considering only mask==True entries.

    Convention:
    - mask: bool tensor broadcastable to x
    - mask==True means "keep this entry"

    Return: same shape as x.mean(dim=dim)

    Requirements:
    - Avoid division by zero: if all mask are False along `dim`, define mean as 0.
    """
    keep = mask.to(x.dtype).expand_as(x)
    total = (x * keep).sum(dim=dim)
    count = keep.sum(dim=dim)
    return torch.where(count > 0, total / count.clamp(min=1), torch.zeros_like(total))

# ---- masked_mean: average only the entries the mask keeps ----
x    = torch.tensor([[1., 2., 3.],
                     [4., 5., 6.]])
mask = torch.tensor([[True,  True,  False],
                     [False, False, False]])

print("x             ", x.tolist())
print("mask          ", [[int(v) for v in r] for r in mask.tolist()], "  1 = keep, 0 = ignore")
print("masked_mean ->", masked_mean(x, mask, dim=1).tolist())
print("               row 0: (1+2)/2 = 1.5   row 1: nothing kept -> 0.0")
print()
print("plain mean -> ", x.mean(dim=1).tolist(), "  wrong: it counts the ignored entries too")

x              [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]]
mask           [[1, 1, 0], [0, 0, 0]]   1 = keep, 0 = ignore
masked_mean -> [1.5, 0.0]
               row 0: (1+2)/2 = 1.5   row 1: nothing kept -> 0.0

plain mean ->  [2.0, 5.0]   wrong: it counts the ignored entries too


## Einsum warmup
Now that you’re comfortable with shapes and broadcasting, we’ll introduce `torch.einsum`, a concise way to express tensor operations by explicitly naming axes and summing over repeated indices.

### The idea
You describe each input tensor by labeling its dimensions with letters, e.g.
- `x: (B, T, D)` → `"btd"`
- `W: (D, H)`    → `"dh"`

Then you tell einsum what output labels you want:
- `"btd,dh->bth"`

### Rules of einsum
1) **Same letter = same axis** (must match in size, except broadcastable size-1).
2) **Repeated letters are summed over** (a “contraction”).
3) **Letters that appear in the output are kept** (in that order).
4) You can **reorder axes** just by changing the output label order.

### Tiny cheat sheet
- Sum over an axis: `"btd->bt"` (sums over `d`)
- Transpose: `"ij->ji"`
- Dot product: `"d,d->"` or batched `"btd,btd->bt"`
- Matrix multiply: `"ik,kj->ij"`
- Batched matmul: `"bij,bjk->bik"`
- Outer product: `"i,j->ij"`

### How to derive an einsum (recommended workflow)
1) Write down shapes with named axes (e.g. `q: b h t d`, `k: b h s d`).
2) Decide which axes you want to **sum over** (give them the same letter in both inputs).
3) Decide which axes you want to **keep** in the output (write them after `->`).

In this section, you’ll use einsum to implement building blocks that show up in attention:
- linear projections (`x @ W`)
- dot products
- attention score matrices (`QKᵀ`)
- applying attention weights (`softmax(scores) @ V`)

NOTE: For these exercises you are required to use `torch.einsum` not `matmul` (we check). You are also not required to understand the attention mechanism at this point and the exercises are sovable without. It is good however, to remember the implementations in this exercise for future implementations.

In [271]:
def einsum_linear_btd_dh_to_bth(x: torch.Tensor, W: torch.Tensor) -> torch.Tensor:
    """
    Linear projection using einsum.

    Shapes:
    - x: (B, T, D)
    - W: (D, H)
    Return:
    - y: (B, T, H)
    """
    return torch.einsum("btd,dh->bth", x, W)

# ---- one weight matrix applied at every (batch, time) position ----
x = torch.tensor([[[1., 2.]]])                  # (B=1, T=1, D=2)
W = torch.tensor([[ 10.,  20.,  30.],           # (D=2, H=3)
                  [100., 200., 300.]])

print("x  (B,T,D) ", x.tolist())
print("W  (D,H)   ", W.tolist())
print("y  (B,T,H) ", einsum_linear_btd_dh_to_bth(x, W).tolist())
print()
print("   y[0,0,0] = 1*10  + 2*100 = 210")
print("   y[0,0,1] = 1*20  + 2*200 = 420")
print("   y[0,0,2] = 1*30  + 2*300 = 630")
print("   the D axis is multiplied and summed away; B, T, H survive")

x  (B,T,D)  [[[1.0, 2.0]]]
W  (D,H)    [[10.0, 20.0, 30.0], [100.0, 200.0, 300.0]]
y  (B,T,H)  [[[210.0, 420.0, 630.0]]]

   y[0,0,0] = 1*10  + 2*100 = 210
   y[0,0,1] = 1*20  + 2*200 = 420
   y[0,0,2] = 1*30  + 2*300 = 630
   the D axis is multiplied and summed away; B, T, H survive


In [272]:
def einsum_pairwise_dot(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """
    Pairwise dot product between x and y.

    Shapes:
    - x: (B, T, D)
    - y: (B, T, D)
    Return:
    - dots: (B, T) where dots[b,t] = dot(x[b,t], y[b,t])
    """
    return torch.einsum("btd,btd->bt", x, y)

# ---- one dot product per (batch, time) position ----
x = torch.tensor([[[1., 2., 3.]]])              # (B=1, T=1, D=3)
y = torch.tensor([[[4., 5., 6.]]])

print("x        ", x.tolist())
print("y        ", y.tolist())
print("dots ->  ", einsum_pairwise_dot(x, y).tolist(), "   1*4 + 2*5 + 3*6 = 32")
print()
# two time positions, so two independent dot products come back
x2 = torch.tensor([[[1., 2.], [3., 4.]]])       # (1, 2, 2)
y2 = torch.tensor([[[10., 20.], [100., 200.]]])
print("x2       ", x2.tolist())
print("y2       ", y2.tolist())
print("dots ->  ", einsum_pairwise_dot(x2, y2).tolist(), "  [1*10+2*20, 3*100+4*200]")
print("           positions are paired up, NOT crossed -- shape (B,T), one number each")

x         [[[1.0, 2.0, 3.0]]]
y         [[[4.0, 5.0, 6.0]]]
dots ->   [[32.0]]    1*4 + 2*5 + 3*6 = 32

x2        [[[1.0, 2.0], [3.0, 4.0]]]
y2        [[[10.0, 20.0], [100.0, 200.0]]]
dots ->   [[50.0, 1100.0]]   [1*10+2*20, 3*100+4*200]
           positions are paired up, NOT crossed -- shape (B,T), one number each


In [273]:
def einsum_qk_scores(q: torch.Tensor, k: torch.Tensor) -> torch.Tensor:
    """
    Compute attention scores QK^T using einsum.

    Shapes:
    - q: (B, H, T, Dh)
    - k: (B, H, T, Dh)
    Return:
    - scores: (B, H, T, T) where scores[b,h,i,j] = dot(q[b,h,i], k[b,h,j])
    """
    return torch.einsum("bhid,bhjd->bhij", q, k)

# ---- score every query token against every key token ----
# each token is a VECTOR. the score of two tokens is their dot product,
# which measures how alike they are.
q = torch.tensor([[[[1., 0.],     # token0 -> points right
                    [1., 0.],     # token1 -> points right too (identical to token0)
                    [0., 1.]]]])  # token2 -> points up
k = q.clone()

print("the three token vectors:")
for i, vec in enumerate(q[0, 0].tolist()):
    print("   token", i, "=", vec)
print()
scores = einsum_qk_scores(q, k)
print("scores[i][j] = token i  DOT  token j")
print("            token0  token1  token2")
for i, row in enumerate(scores[0, 0].tolist()):
    print("   token" + str(i), "     " + "       ".join(str(int(v)) for v in row))
print()
print("   token0 and token1 are the same vector -> score 1, they match")
print("   token0 and token2 are perpendicular   -> score 0, unrelated")
print()
print("check one by hand:  token0 . token1 = 1*1 + 0*0 =", int((q[0,0,0] * k[0,0,1]).sum()))
print("                    token0 . token2 = 1*0 + 0*1 =", int((q[0,0,0] * k[0,0,2]).sum()))
print()
print("shape", tuple(scores.shape), "-- a (T,T) table per head.")
print('in "bhid,bhjd->bhij", i and j are BOTH the token axis, given different')
print("letters so both survive; d is summed away, and that sum IS the dot product.")

the three token vectors:
   token 0 = [1.0, 0.0]
   token 1 = [1.0, 0.0]
   token 2 = [0.0, 1.0]

scores[i][j] = token i  DOT  token j
            token0  token1  token2
   token0      1       1       0
   token1      1       1       0
   token2      0       0       1

   token0 and token1 are the same vector -> score 1, they match
   token0 and token2 are perpendicular   -> score 0, unrelated

check one by hand:  token0 . token1 = 1*1 + 0*0 = 1
                    token0 . token2 = 1*0 + 0*1 = 0

shape (1, 1, 3, 3) -- a (T,T) table per head.
in "bhid,bhjd->bhij", i and j are BOTH the token axis, given different
letters so both survive; d is summed away, and that sum IS the dot product.


In [274]:
def einsum_apply_attention(weights: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    """
    Apply attention weights to values using einsum.

    Shapes:
    - weights: (B, H, T, T)
    - v:       (B, H, T, Dh)
    Return:
    - out:     (B, H, T, Dh) where out[b,h,i] = sum_j weights[b,h,i,j] * v[b,h,j]
    """
    return torch.einsum("bhij,bhjd->bhid", weights, v)

# ---- attention weights blend the value rows together ----
# three value rows, one per token. round numbers so the mixing is easy to follow.
v = torch.tensor([[[[10., 10.],     # value row of token0
                    [20., 20.],     # value row of token1
                    [30., 30.]]]])  # value row of token2

# one row of weights per token: "how much of each value row do I take?"
# each row sums to 1 -- that is what softmax guarantees in real attention.
w = torch.tensor([[[[1.0, 0.0, 0.0],    # token0 takes only row0
                    [0.5, 0.5, 0.0],    # token1 takes half row0 + half row1
                    [0.2, 0.3, 0.5]]]]) # token2 takes a bit of everything

out = einsum_apply_attention(w, v)

print("value rows:")
for i, row in enumerate(v[0, 0].tolist()):
    print("   row", i, "=", row)
print()
print("weights            ->  output           how it was computed")
for i in range(3):
    wi = w[0, 0, i].tolist()
    terms = " + ".join(f"{a:g}*{b:g}" for a, b in zip(wi, [10, 20, 30]) if a != 0)
    print(f"   token{i}  {[f'{a:g}' for a in wi]}  ->  {out[0, 0, i].tolist()}      {terms} = {out[0,0,i,0]:g}")
print()
print("each output row is a WEIGHTED AVERAGE of the value rows.")
print("weight 1.0 on one row = copy it. weights spread out = blend them.")
print()
print('in "bhij,bhjd->bhid": j was "which value row", and summing it away')
print("is exactly the blending. i (which token) and d (features) survive.")

value rows:
   row 0 = [10.0, 10.0]
   row 1 = [20.0, 20.0]
   row 2 = [30.0, 30.0]

weights            ->  output           how it was computed
   token0  ['1', '0', '0']  ->  [10.0, 10.0]      1*10 = 10
   token1  ['0.5', '0.5', '0']  ->  [15.0, 15.0]      0.5*10 + 0.5*20 = 15
   token2  ['0.2', '0.3', '0.5']  ->  [23.0, 23.0]      0.2*10 + 0.3*20 + 0.5*30 = 23

each output row is a WEIGHTED AVERAGE of the value rows.
weight 1.0 on one row = copy it. weights spread out = blend them.

in "bhij,bhjd->bhid": j was "which value row", and summing it away
is exactly the blending. i (which token) and d (features) survive.


## Attention Fundamentals
This exercise introduces some building blocks of the attention mechanism which we will encounter extensively throughout the course. It's not yet required for you to fully understand the mechanism to implement the exercises. However, it's good to remember these building blocks for the future. 

To complete the exercises you should familiarize yourself with these topics:
- Stable softmax read: https://jaykmody.com/blog/stable-softmax/
- Masking: typically this means setting masked logits to -inf *before* softmax.
- For attention: causal masks are upper-triangular (no attending to the future).

In [230]:
def stable_softmax(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
    """
    Numerically stable softmax along `dim`.

    Requirements:
    - Must not overflow for large values in x.
    - Output sums to 1 along `dim`.
    """
    x_max = x.max(dim=dim, keepdim=True).values
    e = torch.exp(x - x_max)
    return e / e.sum(dim=dim, keepdim=True)

# ---- softmax: turn any numbers into probabilities that add up to 1 ----
x = torch.tensor([[1., 2., 3.]])
p = stable_softmax(x, dim=-1)
print("x           ", x.tolist())
print("softmax  -> ", [round(v, 4) for v in p[0].tolist()])
print("sums to     ", p.sum(-1).item(), "   bigger input -> bigger share")
print()
print('the "stable" part: subtracting the max first stops exp() from exploding')
huge = torch.tensor([[1000., 1001., 1002.]])
print("input      ", huge.tolist())
print("naive exp(x) would give inf ->", torch.exp(huge).tolist())
print("stable_softmax handles it   ->", [round(v, 4) for v in stable_softmax(huge, -1)[0].tolist()])

x            [[1.0, 2.0, 3.0]]
softmax  ->  [0.09, 0.2447, 0.6652]
sums to      1.0    bigger input -> bigger share

the "stable" part: subtracting the max first stops exp() from exploding
input       [[1000.0, 1001.0, 1002.0]]
naive exp(x) would give inf -> [[inf, inf, inf]]
stable_softmax handles it   -> [0.09, 0.2447, 0.6652]


In [231]:
def masked_fill_tensor(x: torch.Tensor, mask: torch.Tensor, value: float) -> torch.Tensor:
    """
    Return a copy of x where positions with mask == True are replaced by `value`.
    
    Requirements:
    - mask must be broadcastable to x.
    - do NOT modify x in-place.
    """
    return x.masked_fill(mask, value)

# ---- overwrite the positions the mask marks True ----
x = torch.tensor([[1., 2., 3., 4.]])
mask = torch.tensor([[False, True, False, True]])
print("x             ", x.tolist())
print("mask          ", [int(v) for v in mask[0].tolist()], "  1 = replace this one")
print("filled with 0 ->", masked_fill_tensor(x, mask, 0.0).tolist())
print()
print("x afterwards  ", x.tolist(), "  the original is untouched (not in-place)")

x              [[1.0, 2.0, 3.0, 4.0]]
mask           [0, 1, 0, 1]   1 = replace this one
filled with 0 -> [[1.0, 0.0, 3.0, 0.0]]

x afterwards   [[1.0, 2.0, 3.0, 4.0]]   the original is untouched (not in-place)


In [232]:
def masked_softmax(x: torch.Tensor, mask: torch.Tensor, dim: int = -1) -> torch.Tensor:
    """
    Softmax over x with a boolean mask.

    Convention:
    - mask == True means "invalid and must receive probability 0".
    - Do masking before softmax (i.e., set invalid logits to a large negative).”

    Requirements:
    - Must be numerically stable.
    - Output must be exactly 0 where mask==True.
    - If all entries are masked along `dim`, return all zeros along `dim`.
    - You may reuse functions you implemented above.
    """
    neg_inf = torch.finfo(x.dtype).min
    logits = masked_fill_tensor(x, mask, neg_inf)
    probs = stable_softmax(logits, dim=dim)
    return masked_fill_tensor(probs, mask, 0.0)

# ---- softmax that ignores the masked positions completely ----
x = torch.tensor([[1., 99., 2., 3.]])
mask = torch.tensor([[False, True, False, False]])
print("x               ", x.tolist(), "  99 would normally dominate")
print("mask            ", [int(v) for v in mask[0].tolist()], "   1 = must get probability 0")
p = masked_softmax(x, mask, dim=-1)
print("masked_softmax ->", [round(v, 4) for v in p[0].tolist()])
print("                  the 99 got exactly 0; the rest still sum to", round(p.sum().item(), 4))
print()
all_masked = torch.tensor([[True, True, True, True]])
print("if EVERYTHING is masked ->", masked_softmax(x, all_masked, -1).tolist(), " all zeros, no NaN")

x                [[1.0, 99.0, 2.0, 3.0]]   99 would normally dominate
mask             [0, 1, 0, 0]    1 = must get probability 0
masked_softmax -> [0.09, 0.0, 0.2447, 0.6652]
                  the 99 got exactly 0; the rest still sum to 1.0

if EVERYTHING is masked -> [[0.0, 0.0, 0.0, 0.0]]  all zeros, no NaN


In [233]:
def make_causal_mask(T: int, device: torch.device | str | None = None) -> torch.Tensor:
    """
    Create a causal (future-masking) boolean mask of shape (T, T).

    Convention:
    - mask[i, j] == True  => position (i attends to j) is NOT allowed (j is in the future)
    - mask[i, j] == False => allowed

    So this is an upper-triangular mask above the diagonal.

    Return:
    - mask: boolean tensor on the specified device

    Example (T=4):
        [[F, T, T, T],
         [F, F, T, T],
         [F, F, F, T],
         [F, F, F, F]]
    """
    ones = torch.ones(T, T, dtype=torch.bool, device=device)
    return torch.triu(ones, diagonal=1)

# ---- the causal mask: True marks a position you are NOT allowed to look at ----
m = make_causal_mask(4)
print("make_causal_mask(4)   (1 = blocked, 0 = allowed)")
for i, row in enumerate(m.tolist()):
    print("   token", i, "->", [int(v) for v in row])
print()
print("row i shows what token i may attend to:")
print("   token 0 can only see itself")
print("   token 3 can see everything before it")
print("nobody can see the future -- that is the upper triangle of 1s")

make_causal_mask(4)   (1 = blocked, 0 = allowed)
   token 0 -> [0, 1, 1, 1]
   token 1 -> [0, 0, 1, 1]
   token 2 -> [0, 0, 0, 1]
   token 3 -> [0, 0, 0, 0]

row i shows what token i may attend to:
   token 0 can only see itself
   token 3 can see everything before it
nobody can see the future -- that is the upper triangle of 1s


In [189]:
def apply_causal_mask(attn_logits: torch.Tensor, value: float = -1e9) -> torch.Tensor:
    """
    Apply a causal mask to attention logits.

    Expected shapes:
    - attn_logits: (..., T, T)

    Returns:
    - masked logits (same shape) where masked positions have been set to `value`.

    Notes:
    - Create a causal mask for the final two dims.
    - Broadcast it across leading dims.
    - You may reuse functions declared above.
    """
    mask = make_causal_mask(attn_logits.shape[-1], device=attn_logits.device)
    # clamp so `value` stays representable in low-precision dtypes (e.g. float16)
    fill = max(value, torch.finfo(attn_logits.dtype).min)
    return masked_fill_tensor(attn_logits, mask, fill)

# ---- apply that mask to real attention scores ----
scores = torch.zeros(1, 1, 4, 4)          # start neutral so the mask is easy to see
masked = apply_causal_mask(scores)
print("all scores start at 0. after masking (shown as 0 / -1e9):")
for i, row in enumerate(masked[0, 0].tolist()):
    print("   token", i, "->", "  ".join("0" if v == 0 else "-1e9" for v in row))
print()
p = stable_softmax(masked, dim=-1)
print("after softmax, the -1e9 entries become 0 probability:")
for i, row in enumerate(p[0, 0].tolist()):
    print("   token", i, "->", [round(v, 3) for v in row])
print()
print("token 0 puts all its attention on itself; token 3 spreads it over all four")

all scores start at 0. after masking (shown as 0 / -1e9):
   token 0 -> 0  -1e9  -1e9  -1e9
   token 1 -> 0  0  -1e9  -1e9
   token 2 -> 0  0  0  -1e9
   token 3 -> 0  0  0  0

after softmax, the -1e9 entries become 0 probability:
   token 0 -> [1.0, 0.0, 0.0, 0.0]
   token 1 -> [0.5, 0.5, 0.0, 0.0]
   token 2 -> [0.333, 0.333, 0.333, 0.0]
   token 3 -> [0.25, 0.25, 0.25, 0.25]

token 0 puts all its attention on itself; token 3 spreads it over all four
